# Train SmolVLA with FineTuning

---

- Conda env : [lerobot](../README.md#setup-a-conda-environment)

----

- Ref: 
    - ...


    


### Device Setup

In [1]:
import torch

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Available device : {device}")

Available device : cuda


In [2]:
if device == "cuda":
    !nvidia-smi

Fri Jan 16 14:58:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 22%   38C    P8             30W /  250W |     545MiB /  11264MiB |     15%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## DataSet(svla-so101_pickplace) Visualization

In [3]:
!lerobot-dataset-viz \
    --repo-id lerobot/pusht \
    --episode-index 0

[2026-01-16T22:58:27Z INFO  re_grpc_server] Listening for gRPC connections on 0.0.0.0:9876. Connect by running `rerun --connect rerun+http://127.0.0.1:9876/proxy`
[2026-01-16T22:58:27Z INFO  winit::platform_impl::linux::x11::window] Guessed window scale factor: 1
[2026-01-16T22:58:27Z WARN  wgpu_hal::vulkan::instance] Unable to find extension: VK_EXT_physical_device_drm
[2026-01-16T22:58:27Z WARN  wgpu_hal::gles::egl] No config found!
[2026-01-16T22:58:27Z WARN  wgpu_hal::gles::egl] EGL says it can present to the window but not natively
  0%|                                                     | 0/6 [00:00<?, ?it/s][2026-01-16T22:58:27Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2026-01-16T22:58:27Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2026-01-16T22:58:27Z INFO  egui_wgpu] There were 3 available wgpu adapters: {backend: Vulkan, device_type: DiscreteGpu, name: "NVIDIA GeForce RTX 2080 Ti", 

## Fine-tuning SmolVAL with sval-so101-pickplace dataset

In [4]:
import os

output_dir = os.path.join(os.path.abspath(os.path.curdir), "temp/outputs/pushT")

print(output_dir)

/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/AI_Robotics_Playgrounds/LeRobot_0.4.2/SmolVLA/temp/outputs/pushT


In [5]:
!lerobot-train \
    --policy.path=lerobot/smolvla_base \
    --dataset.repo_id=lerobot/pusht \
    --batch_size=8  \
    --steps=2000 \
    --save_freq=1000 \
    --eval_freq=10 \
    --policy.device=$device \
    --wandb.enable=false \
    --output_dir=./temp/outputs/smolvla_pusht \
    --policy.push_to_hub=false \
    --rename_map='{"observation.image": "observation.images.camera1"}' \
    --policy.empty_cameras=0

INFO 2026-01-16 14:59:27 ot_train.py:163 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         't

In [18]:
!lerobot-eval \
    --policy.type=smolvla \
    --policy.pretrained_path=./temp/outputs/smolvla_pusht/checkpoints/last/pretrained_model \
    --env.type=pusht \
    --eval.n_episodes=5 \
    --eval.batch_size=1 \
    --policy.device=$device \
    --env.render_mode=rgb_array \
    --output_dir=./temp/outputs/smolvla_pusht/eval

WARNING 2026-01-16 15:54:44 figs/eval.py:51 No pretrained path was provided, evaluated policy will be built from scratch (random weights).
WARNING 2026-01-16 15:54:44 figs/eval.py:63 No job name provided, using 'pusht_smolvla' as job name.
INFO 2026-01-16 15:54:44 bot_eval.py:499 {'env': {'disable_env_checker': True,
         'episode_length': 300,
         'features': {'action': {'shape': (2,),
                                 'type': <FeatureType.ACTION: 'ACTION'>},
                      'agent_pos': {'shape': (2,),
                                    'type': <FeatureType.STATE: 'STATE'>},
                      'pixels': {'shape': (384, 384, 3),
                                 'type': <FeatureType.VISUAL: 'VISUAL'>}},
         'features_map': {'action': 'action',
                          'agent_pos': 'observation.state',
                          'environment_state': 'observation.environment_state',
                          'pixels': 'observation.image'},
         'fps': 10,
     